# Adaptive Adversarial Threat Detection and Defense Framework
## Notebook 1 — Baseline Model (Week 4)

**Covers:** Preprocessing → Training → Evaluation → Visualizations

**Before running:** Upload `KDDTrain+.txt` and `KDDTest+.txt` to Colab (Files panel → Upload).

In [ ]:
# ─── Cell 1: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
print('All imports successful.')

In [ ]:
# ─── Cell 2: Column names ──────────────────────────────────────────────────────
# NSL-KDD has no header row — we supply 43 column names manually.
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]
print(f'Column count: {len(columns)} (expected 43)')

In [ ]:
# ─── Cell 3: Load data ─────────────────────────────────────────────────────────
train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df  = pd.read_csv('KDDTest+.txt',  names=columns)
print('Train:', train_df.shape, '| Test:', test_df.shape)
train_df.head(3)

In [ ]:
# ─── Cell 4: Drop difficulty column ───────────────────────────────────────────
# 'difficulty' is a meta-attribute, not a network traffic feature.
train_df = train_df.drop('difficulty', axis=1)
test_df  = test_df.drop('difficulty', axis=1)
print('After drop — Train:', train_df.shape, '| Test:', test_df.shape)

In [ ]:
# ─── Cell 5: Create binary labels ─────────────────────────────────────────────
# normal=0, all attack types=1
train_df['label'] = train_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['label']  = test_df['label'].apply(lambda x: 0 if x == 'normal' else 1)
print('Train label distribution:')
print(train_df['label'].value_counts())
print('\nTest label distribution:')
print(test_df['label'].value_counts())

In [ ]:
# ─── Cell 6: Encode categorical features ──────────────────────────────────────
# Fit LabelEncoder on combined train+test values to handle all categories seen
# in either split (test set contains some service values not in training set).
cat_cols = ['protocol_type', 'service', 'flag']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined_vals = pd.concat([train_df[col], test_df[col]])
    le.fit(combined_vals)
    train_df[col] = le.transform(train_df[col])
    test_df[col]  = le.transform(test_df[col])
    encoders[col] = le
print('Categorical encoding complete.')

In [ ]:
# ─── Cell 7: Separate features and labels ─────────────────────────────────────
# Use NSL-KDD's pre-defined train/test split (do not apply train_test_split).
feature_cols = [c for c in train_df.columns if c != 'label']

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test  = test_df[feature_cols].values
y_test  = test_df['label'].values

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'y_train: {y_train.shape} | y_test: {y_test.shape}')

In [ ]:
# ─── Cell 8: Normalize with StandardScaler ────────────────────────────────────
# IMPORTANT: fit ONLY on X_train to prevent data leakage.
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit + transform training set
X_test  = scaler.transform(X_test)       # transform only — no re-fitting

print(f'Scaling complete.')
print(f'X_train mean (should be ~0): {X_train.mean():.6f}')
print(f'X_train std  (should be ~1): {X_train.std():.6f}')

In [ ]:
# ─── Cell 9: Preprocessing verification ───────────────────────────────────────
print('=== Preprocessing Complete ===')
print(f'X_train: {X_train.shape}  (expected ~125973 x 41)')
print(f'X_test:  {X_test.shape}   (expected ~22544 x 41)')
print(f'Train class balance — 0 (normal): {(y_train==0).sum()} | 1 (attack): {(y_train==1).sum()}')
print(f'Test class balance  — 0 (normal): {(y_test==0).sum()}  | 1 (attack): {(y_test==1).sum()}')
print(f'No null values: {np.isnan(X_train).sum() + np.isnan(X_test).sum() == 0}')

---
## Condition 1 — Baseline Model Training and Evaluation

In [ ]:
# ─── Cell 10: Train baseline Random Forest ────────────────────────────────────
# n_jobs=-1 uses all CPU cores to speed up training.
# random_state=42 ensures reproducibility.
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_baseline.fit(X_train, y_train)
print('Training complete.')

In [ ]:
# ─── Cell 11: Evaluate on clean test set (Condition 1 — Baseline) ─────────────
y_pred_baseline = rf_baseline.predict(X_test)
y_prob_baseline = rf_baseline.predict_proba(X_test)[:, 1]  # probability of attack class

acc_c1  = accuracy_score(y_test, y_pred_baseline)
prec_c1 = precision_score(y_test, y_pred_baseline)
rec_c1  = recall_score(y_test, y_pred_baseline)
f1_c1   = f1_score(y_test, y_pred_baseline)
auc_c1  = roc_auc_score(y_test, y_prob_baseline)

print('=== CONDITION 1: BASELINE RESULTS ===')
print(f'Accuracy : {acc_c1:.4f}')
print(f'Precision: {prec_c1:.4f}')
print(f'Recall   : {rec_c1:.4f}')
print(f'F1-Score : {f1_c1:.4f}')
print(f'AUC-ROC  : {auc_c1:.4f}')

In [ ]:
# ─── Cell 12: Feature importances — identify top 5 for adversarial attack ─────
feature_names = feature_cols  # list of 41 feature names (without 'label')
importances   = rf_baseline.feature_importances_
top10_idx     = np.argsort(importances)[::-1][:10]
top5_idx      = top10_idx[:5]   # SAVE THIS — used in Week 5 attack

print('Top 10 features by importance:')
for rank, idx in enumerate(top10_idx, 1):
    print(f'  {rank:2}. {feature_names[idx]:35s} {importances[idx]:.4f}')

print(f'\nTop 5 feature indices (for Week 5 attack): {top5_idx}')
print(f'Top 5 feature names: {[feature_names[i] for i in top5_idx]}')

---
## Baseline Visualizations

In [ ]:
# ─── Cell 13: Confusion matrix ────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_baseline)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title('Confusion Matrix — Condition 1: Baseline')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion-matrix-baseline.png', dpi=150)
plt.show()
print('Saved: confusion-matrix-baseline.png')

In [ ]:
# ─── Cell 14: ROC curve ───────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob_baseline)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2,
         label=f'Baseline ROC (AUC = {auc_c1:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — Condition 1: Baseline')
plt.legend()
plt.tight_layout()
plt.savefig('roc-baseline.png', dpi=150)
plt.show()
print('Saved: roc-baseline.png')

In [ ]:
# ─── Cell 15: Feature importance bar chart ────────────────────────────────────
top10_names  = [feature_names[i] for i in top10_idx]
top10_values = importances[top10_idx]

plt.figure(figsize=(8, 5))
plt.barh(top10_names[::-1], top10_values[::-1], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Top 10 Feature Importances — Random Forest Baseline')
plt.tight_layout()
plt.savefig('feature-importance-baseline.png', dpi=150)
plt.show()
print('Saved: feature-importance-baseline.png')

In [ ]:
# ─── Cell 16: Results summary table ──────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════╗')
print('║        WEEK 4 RESULTS — COPY THESE TO results-baseline.md   ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Accuracy : {acc_c1:.4f}                                        ║')
print(f'║  Precision: {prec_c1:.4f}                                        ║')
print(f'║  Recall   : {rec_c1:.4f}                                        ║')
print(f'║  F1-Score : {f1_c1:.4f}                                        ║')
print(f'║  AUC-ROC  : {auc_c1:.4f}                                        ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Top 5 feature indices: {top5_idx}         ║')
print(f'║  Top 5 feature names:                                       ║')
for i, idx in enumerate(top5_idx):
    print(f'║    {i+1}. {feature_names[idx]:52s}║')
print('╚══════════════════════════════════════════════════════════════╝')